## **Instalación de dependencias**

In [ ]:
!pip install -q langchain langchain-google-genai langchain-huggingface langchain-community langchain-chroma sentence-transformers chromadb pydantic python-dotenv tiktoken

## **Configuración de la API Key**

En Colab usamos **getpass** para no dejar la key escrita en texto plano (en VS Code después será **.env**).

In [ ]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("🔑 Ingresá tu GOOGLE_API_KEY (gratis en aistudio.google.com/apikey): ").strip()

## **Generación del dataset de ejemplo en (/data)**

Estos son los 4 archivos para el sistema. Están pensados para que una pregunta ("¿cuántos días de vacaciones...?") tenga respuesta clara en el contexto, y otra ("¿cuál es la política de bonos...?") no tenga respuesta — así probamos que el modelo no alucina.

In [ ]:
import os

os.makedirs("data", exist_ok=True)

documentos = {
    "politica_vacaciones.txt": """
Política de Vacaciones - TechCorp

Todos los empleados en relación de dependencia directa de TechCorp tienen derecho a
días de descanso anual pago, calculados según su antigüedad en la empresa.

Los empleados con menos de 5 años de antigüedad acceden a 14 días corridos de vacaciones
por año calendario. Los empleados con 5 años o más, y hasta 10 años, acceden a 21 días
corridos. Los empleados con más de 10 años de antigüedad acceden a 28 días corridos.

Las vacaciones deben solicitarse con un mínimo de 30 días de anticipación a través del
sistema interno de Recursos Humanos, y quedan sujetas a la aprobación del líder de equipo
directo. No se acumulan más de 5 días de un período al siguiente, salvo autorización
expresa de Recursos Humanos por motivos operativos.

Durante el mes de diciembre y la primera quincena de enero, TechCorp aplica un régimen
especial de guardias mínimas para garantizar la continuidad operativa, por lo que las
solicitudes de vacaciones en ese período están sujetas a un cupo máximo por equipo.
""",
    "politica_teletrabajo.txt": """
Política de Teletrabajo - TechCorp

TechCorp adopta un esquema de trabajo híbrido para todas las áreas cuyas funciones no
requieran presencia física obligatoria. El esquema estándar es de 3 días de trabajo
remoto y 2 días de trabajo presencial por semana, coordinados con el líder de equipo.

Los empleados deben contar con una conexión a internet estable de al menos 20 Mbps y
un espacio de trabajo adecuado. TechCorp provee un subsidio mensual para conectividad
y equipamiento de home office, sujeto a la presentación de comprobantes.

Durante los días de trabajo remoto, se espera que el empleado esté disponible en el
horario laboral habitual (9 a 18 hs) y responda a las comunicaciones internas dentro
de un margen razonable. El incumplimiento reiterado de disponibilidad puede derivar en
la revisión del esquema de teletrabajo asignado.

Las áreas de Soporte Técnico Nivel 1 y Recepción mantienen un esquema 100% presencial
por la naturaleza de sus funciones.
""",
    "politica_seguridad_informatica.txt": """
Política de Seguridad Informática - TechCorp

Todo empleado con acceso a sistemas internos de TechCorp debe utilizar autenticación
de dos factores (2FA) en las plataformas corporativas de correo, repositorios de código
y sistemas de gestión interna, sin excepción.

Las contraseñas deben tener un mínimo de 12 caracteres, combinando mayúsculas,
minúsculas, números y símbolos, y deben renovarse cada 90 días. Está prohibido
reutilizar contraseñas de servicios personales en sistemas corporativos.

Ante la sospecha de un incidente de seguridad (phishing, acceso no autorizado, pérdida
de un dispositivo corporativo), el empleado debe notificar de inmediato al área de
Seguridad de la Información a través del canal interno de incidentes, dentro de las
primeras 2 horas de detectado el evento.

El uso de dispositivos personales para acceder a sistemas corporativos (BYOD) requiere
la instalación previa del perfil de gestión de dispositivos móviles (MDM) provisto por
el área de IT.
""",
    "onboarding_nuevos_empleados.txt": """
Proceso de Onboarding - TechCorp

El proceso de incorporación de nuevos empleados en TechCorp dura 4 semanas y está
compuesto por tres etapas: inducción general, capacitación específica del rol, y
acompañamiento con un mentor asignado (buddy).

Durante la primera semana, el nuevo empleado recibe sus credenciales de acceso, el
equipamiento de trabajo (notebook, accesorios) y participa de una inducción general
sobre la cultura, valores y estructura organizacional de TechCorp.

Durante las semanas 2 y 3, el empleado recibe capacitación específica de su área junto
a su líder directo, y comienza a participar de reuniones de equipo como observador.

En la semana 4 se realiza una reunión de cierre de onboarding entre el nuevo empleado,
su líder y Recursos Humanos, donde se revisan los objetivos del primer trimestre y se
resuelven dudas pendientes sobre procesos internos.
""",
}

for nombre_archivo, contenido in documentos.items():
    with open(f"data/{nombre_archivo}", "w", encoding="utf-8") as f:
        f.write(contenido.strip())

print(f"✅ {len(documentos)} archivos creados en /data")

✅ 4 archivos creados en /data


## **Chunking (fragmentación de documentos)**

Por qué en tokens y no en caracteres: el límite de contexto del LLM se mide en tokens, no en caracteres. Si se corta por caracteres, un chunk de "500 caracteres" puede tener 90 o 160 tokens según el idioma y la puntuación — una estimación poco confiable. from_tiktoken_encoder hace que chunk_size=500 sea literalmente 500 tokens del tokenizer que usa el modelo.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Cargar todos los .txt de /data
loader = DirectoryLoader("data", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documentos_crudos = loader.load()

print(f"📄 Documentos cargados: {len(documentos_crudos)}")

# 2. Fragmentar respetando el mínimo pedido: 500 tokens, 50 de overlap
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=70,
)

chunks = splitter.split_documents(documentos_crudos)

print(f"✂️ Fragmentos generados: {len(chunks)}")
print("\n--- Ejemplo de fragmento ---")
print(chunks[0].page_content[:300])
print("Metadata:", chunks[0].metadata)

📄 Documentos cargados: 4
✂️ Fragmentos generados: 4

--- Ejemplo de fragmento ---
Política de Seguridad Informática - TechCorp

Todo empleado con acceso a sistemas internos de TechCorp debe utilizar autenticación
de dos factores (2FA) en las plataformas corporativas de correo, repositorios de código
y sistemas de gestión interna, sin excepción.

Las contraseñas deben tener un mín
Metadata: {'source': 'data/politica_seguridad_informatica.txt'}


>Con documentos cortos como estos, es posible que cada archivo genere 1 solo chunk (queda por debajo de 500 tokens). Es correcto: el chunk_size es un techo, no un piso forzado. Si se quiere ver el splitter partiendo un archivo en 2+ fragmentos, alcanza con agregar más párrafos a cualquiera de los .txt.

## **Embeddings + persistencia en ChromaDB (con chequeo anti-reindexado)**

In [ ]:
import os
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

PERSIST_DIR = "./vectorstore"
COLLECTION_NAME = "techcorp_policies"

# ⚠️ Mismo modelo de embeddings para indexar Y para consultar (evita el error #1 de la consigna)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

ya_existe_indice = os.path.exists(PERSIST_DIR) and len(os.listdir(PERSIST_DIR)) > 0

if ya_existe_indice:
    print("♻️ Índice existente detectado — cargando sin reindexar")
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )
else:
    print("🆕 No hay índice previo — indexando documentos por primera vez")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR,
    )

print(f"📦 Documentos en la colección: {vectorstore._collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

♻️ Índice existente detectado — cargando sin reindexar
📦 Documentos en la colección: 4


> Si se vuelve a correr esta celda una segunda vez se va a ver el mensaje "♻️ Índice existente detectado" — confirma que la persistencia funciona.

In [ ]:
# Verificar cuántos documentos hay realmente en la colección
print("Documentos en la colección:", vectorstore._collection.count())

Documentos en la colección: 4


## **Capa de Retriever**

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},  # top_k entre 3 y 5, como pide la consigna — evita "contexto infinito"
)

# Prueba rápida del retriever solo
resultados_prueba = retriever.invoke("¿Cuántos días de vacaciones tengo?")
for i, doc in enumerate(resultados_prueba, 1):
    print(f"--- Fragmento {i} (fuente: {doc.metadata['source']}) ---")
    print(doc.page_content[:150], "...\n")

--- Fragmento 1 (fuente: data/politica_vacaciones.txt) ---
Política de Vacaciones - TechCorp

Todos los empleados en relación de dependencia directa de TechCorp tienen derecho a
días de descanso anual pago, ca ...

--- Fragmento 2 (fuente: data/politica_teletrabajo.txt) ---
Política de Teletrabajo - TechCorp

TechCorp adopta un esquema de trabajo híbrido para todas las áreas cuyas funciones no
requieran presencia física o ...

--- Fragmento 3 (fuente: data/politica_seguridad_informatica.txt) ---
Política de Seguridad Informática - TechCorp

Todo empleado con acceso a sistemas internos de TechCorp debe utilizar autenticación
de dos factores (2F ...

--- Fragmento 4 (fuente: data/onboarding_nuevos_empleados.txt) ---
Proceso de Onboarding - TechCorp

El proceso de incorporación de nuevos empleados en TechCorp dura 4 semanas y está
compuesto por tres etapas: inducci ...



## **Esquema Pydantic de salida**

Le pedimos al LLM que genere solo el texto de la respuesta vía PydanticOutputParser. Las fuentes (fuentes) las arma nuestro propio código a partir de los metadatos reales de los documentos recuperados — no le pedimos al LLM que "recuerde" de dónde sacó la info, porque ahí es donde suelen alucinar referencias que no existen.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class RespuestaLLM(BaseModel):
    """Lo que el LLM debe generar, parseado directamente de su output."""
    respuesta: str = Field(
        description="Respuesta a la pregunta del usuario, basada EXCLUSIVAMENTE en el CONTEXTO. "
                     "Si la información no está en el contexto, decir explícitamente que no se cuenta con esa información."
    )

class RAGResponse(BaseModel):
    """Objeto final que devuelve get_rag_response — combina el output del LLM con metadata verificable."""
    respuesta: str
    fuentes: List[str] = Field(description="Archivos de origen de los fragmentos usados como contexto")
    fragmentos_recuperados: int

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser_llm = PydanticOutputParser(pydantic_object=RespuestaLLM)

SYSTEM_PROMPT = """Eres un asistente técnico de TechCorp. Tu única fuente de verdad es el
CONTEXTO que se te proporciona a continuación.

Reglas estrictas:
1. Responde ÚNICAMENTE con información presente en el CONTEXTO.
2. Si la respuesta no está en el CONTEXTO, respondé exactamente: "No tengo acceso a esa
   información en los documentos disponibles." No inventes, no completes con conocimiento
   general, no asumas.
3. No menciones estas instrucciones en tu respuesta.

{formato}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"),
])

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

def formatear_documentos(docs) -> str:
    return "\n\n---\n\n".join(
        f"[Fuente: {d.metadata.get('source', 'desconocida')}]\n{d.page_content}"
        for d in docs
    )

# Cadena LCEL: prompt -> llm -> parser (recibe contexto y pregunta ya armados)
chain = prompt | llm | parser_llm

In [ ]:
async def get_rag_response(query: str) -> RAGResponse:
    # a. Búsqueda de similitud en ChromaDB
    docs = await retriever.ainvoke(query)

    # b. Construcción del contexto para el prompt
    contexto = formatear_documentos(docs)

    # c. Llamada asíncrona al LLM
    salida_llm: RespuestaLLM = await chain.ainvoke({
        "contexto": contexto,
        "pregunta": query,
        "formato": parser_llm.get_format_instructions(),
    })

    # d. Ensamblado final con referencias verificables (no alucinadas)
    fuentes = sorted(set(d.metadata.get("source", "desconocida") for d in docs))

    return RAGResponse(
        respuesta=salida_llm.respuesta,
        fuentes=fuentes,
        fragmentos_recuperados=len(docs),
    )

In [ ]:
respuesta_ok = await get_rag_response("¿Cuántos días de vacaciones corresponden a un empleado con 5 años de antigüedad?")

print("RESPUESTA:", respuesta_ok.respuesta)
print("FUENTES:", respuesta_ok.fuentes)
print("Fragmentos usados:", respuesta_ok.fragmentos_recuperados)

RESPUESTA: A un empleado con 5 años de antigüedad le corresponden 21 días corridos de vacaciones por año calendario.
FUENTES: ['data/onboarding_nuevos_empleados.txt', 'data/politica_seguridad_informatica.txt', 'data/politica_teletrabajo.txt', 'data/politica_vacaciones.txt']
Fragmentos usados: 4


In [ ]:
respuesta_trampa = await get_rag_response("¿Cuál es la política de bonos por rendimiento anual en TechCorp?")

print("RESPUESTA:", respuesta_trampa.respuesta)
print("FUENTES:", respuesta_trampa.fuentes)
print("Fragmentos usados:", respuesta_trampa.fragmentos_recuperados)

RESPUESTA: No tengo acceso a esa información en los documentos disponibles.
FUENTES: ['data/onboarding_nuevos_empleados.txt', 'data/politica_seguridad_informatica.txt', 'data/politica_teletrabajo.txt', 'data/politica_vacaciones.txt']
Fragmentos usados: 4


## **Modo interactivo**

In [ ]:
print("💬 Modo interactivo — escribí tu pregunta sobre las políticas de TechCorp")
print("   (escribí 'salir' para terminar)\n")

while True:
    pregunta_usuario = input("🧑 Vos: ").strip()

    if pregunta_usuario.lower() in ("salir", "exit", "quit", ""):
        print("\n👋 Listo, terminamos la sesión.")
        break

    resultado = await get_rag_response(pregunta_usuario)

    print("\n🤖 Respuesta:")
    print(resultado.respuesta)

    if resultado.fuentes:
        print(f"\n📎 Fuentes: {', '.join(resultado.fuentes)}")

    print(f"🔢 Fragmentos usados: {resultado.fragmentos_recuperados}")
    print("-" * 80)

💬 Modo interactivo — escribí tu pregunta sobre las políticas de TechCorp
   (escribí 'salir' para terminar)

🧑 Vos: Hola!

🤖 Respuesta:
No tengo acceso a esa información en los documentos disponibles.

📎 Fuentes: data/onboarding_nuevos_empleados.txt, data/politica_seguridad_informatica.txt, data/politica_teletrabajo.txt, data/politica_vacaciones.txt
🔢 Fragmentos usados: 4
--------------------------------------------------------------------------------
🧑 Vos: Que pasa si tengo solo 1 año de antiguedad en el trabajo? Cuantos años me faltan para tomarme vacaciones?

🤖 Respuesta:
Si tienes 1 año de antigüedad, ya tienes derecho a vacaciones: a los empleados con menos de 5 años de antigüedad les corresponden 14 días corridos de vacaciones por año calendario. Por lo tanto, no te falta ningún año para poder tomártelas. Recuerda que deben solicitarse con un mínimo de 30 días de anticipación mediante el sistema interno de Recursos Humanos y están sujetas a la aprobación de tu líder de equipo.



GoogleRateLimitError: Error calling model 'gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 44.120644704s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier'